In [1]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add the project root directory to sys.path to enable local module imports
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import custom data cleaning and feature engineering pipelines
from source.financial_cleaning import (
    clean_financial_pipeline,
    split_datasets,
)
from source.financial_feature_engineering import feature_engineering_pipeline

print("Libraries & Custom Modules imported successfully!")

Libraries & Custom Modules imported successfully!


In [3]:
# Define paths relative to the project root
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
DIVAR_CSV_FILE = RAW_DATA_PATH / "Divar.csv"

# Load the primary raw Divar dataset
if not DIVAR_CSV_FILE.exists():
    raise FileNotFoundError(f"Divar.csv not found at {DIVAR_CSV_FILE}")

df_raw = pd.read_csv(DIVAR_CSV_FILE)
print(f"Loaded Raw Dataset: {DIVAR_CSV_FILE.name} | Shape: {df_raw.shape}")

# Inspect initial records
df_raw.head(3)

C:\Users\98936\AppData\Local\Temp\ipykernel_39324\886140878.py:9: DtypeWarning: Columns (0: rent_to_single, 1: floor, 2: total_floors_count, 3: extra_person_capacity) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(DIVAR_CSV_FILE)


Loaded Raw Dataset: Divar.csv | Shape: (1000000, 61)


,Unnamed: 0,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,...,property_type,regular_person_capacity,extra_person_capacity,cost_per_extra_person,rent_price_on_regular_days,rent_price_on_special_days,rent_price_at_weekends,location_latitude,location_longitude,location_radius
0,0,temporary-rent,villa,karaj,mehrshahr,2024-08-01 00:00:00,مشاور املاک,۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه,NaN,...,NaN,4.0,6,350000.0,1500000.0,3.500000e+09,3500000.0,35.811684,50.936600,500.0
1,1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0
2,2,residential-rent,apartment-rent,tehran,tohid,2024-10-01 00:00:00,NaN,تخلیه پایان ماه,آپارتمان ۳ خوابه ۱۳۲ متر,مقطوع,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.703865,51.373459,NaN


In [4]:
from source.financial_cleaning import (
    add_business_flags,
    add_deal_type,
    add_full_credit_equivalent,
    clean_text_columns,
    drop_dead_columns,
    remove_impossible_values,
    winsorize_targets,
)

# ==========================================
# Step 1: Core Financial Cleaning
# ==========================================
print("--- [Step 1: Running Financial Cleaning] ---")
df_step1 = clean_text_columns(df_raw)
df_step1 = add_deal_type(df_step1)
df_step1 = add_business_flags(df_step1)
df_step1 = remove_impossible_values(df_step1, verbose=True)
df_step1 = winsorize_targets(df_step1, verbose=True)
df_step1 = add_full_credit_equivalent(df_step1)

# Preserve daily rentals before dropping redundant columns later
df_daily = (
    df_step1[df_step1["rent_price_on_regular_days"].notna()].copy()
    if "rent_price_on_regular_days" in df_step1.columns
    else df_step1.iloc[0:0].copy()
)

# ==========================================
# Step 2: Financial Feature Engineering
# ==========================================
print("\n--- [Step 2: Running Feature Engineering] ---")
df_featured = feature_engineering_pipeline(df_step1, verbose=True)

# ==========================================
# Step 3: Drop Dead / Redundant Columns
# ==========================================
print("\n--- [Step 3: Dropping Redundant Columns] ---")
df_cleaned = drop_dead_columns(df_featured, verbose=True)

print(f"\n Pipeline Finished Successfully! Cleaned Dataset Shape: {df_cleaned.shape}")

--- [Step 1: Running Financial Cleaning] ---
[Data Cleaning] Impossible values mapped to NaN:
  - price_value: 13,186 rows
  - credit_value: 3,109 rows
  - rent_value: 489 rows
  - Winsorized price_value: 5,453 rows capped at 85,000,000,000
  - Winsorized credit_value: 2,967 rows capped at 5,000,000,000
  - Winsorized rent_value: 3,167 rows capped at 200,000,000

--- [Step 2: Running Feature Engineering] ---
[Financial Feature Engineering] Successfully added features: ['equivalent_full_credit', 'weekend_price_ratio', 'special_day_price_ratio', 'is_rent_credit_convertible', 'allows_single_tenant', 'log_price_value', 'log_rent_value', 'log_credit_value']

--- [Step 3: Dropping Redundant Columns] ---
[Data Cleaning] Dropping dead/redundant columns: ['rent_to_single', 'rent_type', 'rent_price_on_regular_days', 'rent_price_on_special_days', 'rent_price_at_weekends', 'transformable_price', 'transformable_credit', 'transformable_rent', 'transformed_credit', 'transformed_rent', 'rent_credit_tr

In [7]:
# =========================================================
# Helper Function for Safe Parquet Export
# =========================================================
def save_parquet_safe(df, file_path):
    """Cast object columns to string before saving to avoid PyArrow mixed-type errors."""
    df_to_save = df.copy()
    for col in df_to_save.select_dtypes(include=["object"]).columns:
        df_to_save[col] = df_to_save[col].astype("string")
    df_to_save.to_parquet(file_path, index=False)


# =========================================================
# Step 4: Split Datasets for Analysis (EDA) and Training
# =========================================================
df_sale, df_rent, train_sale, train_rent = split_datasets(df_cleaned)

print("=" * 50)
print("Dataset Splitting Summary:")
print("=" * 50)
print(f"All Sale Listings (EDA):       {len(df_sale):,} rows")
print(f"All Rent Listings (EDA):       {len(df_rent):,} rows")
print(f"Clean Sale Training (Models): {len(train_sale):,} rows")
print(f"Clean Rent Training (Models): {len(train_rent):,} rows")
print(f"Daily Rentals Subset:         {len(df_daily):,} rows")
print("=" * 50)

# =========================================================
# Step 5: Save Processed Outputs to Parquet
# =========================================================
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
SPLITS_PATH = PROJECT_ROOT / "data" / "splits"

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
SPLITS_PATH.mkdir(parents=True, exist_ok=True)

# 1. Save unified cleaned dataset
save_parquet_safe(df_cleaned, PROCESSED_PATH / "df_processed.parquet")

# 2. Save EDA subsets (including negotiable listings)
save_parquet_safe(df_sale, SPLITS_PATH / "sale_all.parquet")
save_parquet_safe(df_rent, SPLITS_PATH / "rent_all.parquet")

# 3. Save clean training sets (ready for ML regression models)
save_parquet_safe(train_sale, SPLITS_PATH / "sale_train_ready.parquet")
save_parquet_safe(train_rent, SPLITS_PATH / "rent_train_ready.parquet")

# 4. Save daily rentals if available
if not df_daily.empty:
    save_parquet_safe(df_daily, SPLITS_PATH / "daily_rent.parquet")

print("\nAll datasets saved successfully to:")
print(f" - {PROCESSED_PATH}")
print(f" - {SPLITS_PATH}")

Dataset Splitting Summary:
All Sale Listings (EDA):       573,606 rows
All Rent Listings (EDA):       352,994 rows
Clean Sale Training (Models): 555,160 rows
Clean Rent Training (Models): 351,080 rows
Daily Rentals Subset:         18,068 rows

All datasets saved successfully to:
 - C:\Users\98936\Desktop\divar-housing-analysis\data\processed
 - C:\Users\98936\Desktop\divar-housing-analysis\data\splits


C:\Users\98936\AppData\Local\Temp\ipykernel_39324\3922019620.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_to_save.select_dtypes(include=["object"]).columns:
